
# Revision 08 — Paper Figure 5: selected K=46 spatial MS-AA archetypes (summary-layout glass-brain version)

This notebook builds the revised **main-text Figure 5** from the final held-out-selected spatial archetypes.

## Locked analysis choices

- spatial MS-AA
- across-condition shared fit
- representative model order: **K = 46**
- sparse subset size: **top-5**
- selected archetypes: **[4, 5, 6, 12, 8]**

These archetype IDs are treated as **preselected from the held-out cross-fitted analysis** and are never changed based on their appearance.

## Revised figure layout

Each row is one selected archetype:

1. **Mean temporal motif** — cross-participant mean `sXC[:, k]` ± SEM.
2. **Spatial deployment** — participant-averaged `S[k, :]`, shown on HTFA node coordinates and colored by Yeo-7 network. A threshold is used **only for display**.
3. **Yeo-7 network composition** — fraction of the archetype's **absolute spatial coefficient mass across all 700 nodes** assigned to each network. No display threshold is used here.
4. **Cross-fitted selection stability** — how often the archetype entered the held-out top-5 set in intact, word-scrambled, and rest.

This replaces the older exploratory network-composition figure with a single interpretation figure tied directly to the held-out-selected motifs.

### Important interpretation note

The Yeo bars are **descriptive network composition**, not network enrichment. Claims of enrichment should continue to rely on the separate network-size-preserving null analysis.


### Layout change in this revision

This version gives the **glass-brain spatial deployment plots substantially more space**.

- The **main upper block** is a **5 × 2 layout**:
  - column 1 = temporal motif
  - column 2 = large glass-brain spatial deployment

- The **bottom summary block** contains:
  - a **stacked horizontal bar chart** showing Yeo-7 composition across all selected archetypes
  - a **heatmap** showing cross-fitted top-5 selection frequency across intact / word / rest

This layout makes the brain plots the visual focus while still preserving direct comparison of network composition and selection stability.


In [ ]:

# ============================================================
# SETTINGS
# ============================================================
from pathlib import Path

ANALYSIS_TYPE = "spatial"
FIT_SCOPE = "across"
K = 46
TOP_M = 5
SELECTED_ARCHETYPES = [4, 5, 6, 12, 8]
CONDITIONS = ["intact", "word", "rest"]
COND_COLORS = {"intact": "purple", "word": "green", "rest": "0.45"}

FIT_DIR = Path("msaa_flexible_outputs_npz")
FIT_FILE = FIT_DIR / f"spatialAA_across_acrossCond_K{K}.npz"

HELDOUT_DIR = Path(f"msaa_heldout_topm_outputs_{ANALYSIS_TYPE}_{FIT_SCOPE}")
HELDOUT_RUN_CSV = HELDOUT_DIR / "heldout_topm_crossfit_runs.csv"
HELDOUT_STABILITY_CSV = HELDOUT_DIR / "heldout_archetype_selection_stability.csv"

POSTERIOR_MAT = Path("data/pieman/raw/pieman_posterior_K700.mat")
SCHAEFER_TXT = Path("data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order.txt")
SCHAEFER_NII = Path("data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz")

DISPLAY_THR_FRAC = 0.30   # display only
NODE_SIZE_MIN = 10
NODE_SIZE_MAX = 70
FIGSIZE = (16, 15)
DPI = 300

OUTPUT_DIR = Path("revision_8_figure5_selected_K46_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_PNG = OUTPUT_DIR / "Figure5_selected_K46_archetypes_summary_layout.png"
FIGURE_PDF = OUTPUT_DIR / "Figure5_selected_K46_archetypes_summary_layout.pdf"

print("Fit:", FIT_FILE)
print("Held-out runs:", HELDOUT_RUN_CSV)
print("Output:", OUTPUT_DIR.resolve())


In [ ]:

%matplotlib inline

import ast as pyast
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from IPython.display import display

try:
    import nibabel as nib
    from nilearn.input_data import NiftiMasker
    from nilearn import plotting as niplot
    NILEARN_AVAILABLE = True
except Exception:
    NILEARN_AVAILABLE = False
    print("nibabel/nilearn unavailable; Yeo mapping cannot be recomputed.")


## 1. Load the locked K=46 fit

In [ ]:

def to_float_array(x):
    return np.asarray(x, dtype=float)


def load_msaa_npz(path):
    data = np.load(path, allow_pickle=True)
    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()

    out = {
        "K": int(np.asarray(data["K"]).item()),
        "results_subj": results_subj,
        "condition_labels_str": np.asarray(data["condition_labels_str"]).astype(str),
    }
    for key in ["condition_codes", "condition_names", "centers", "widths", "C", "analysis_type", "fit_scope"]:
        if key in data:
            out[key] = data[key]
    return out


if not FIT_FILE.exists():
    raise FileNotFoundError(f"Missing fit file: {FIT_FILE}")

fit = load_msaa_npz(FIT_FILE)
assert fit["K"] == K, (fit["K"], K)
results_subj = fit["results_subj"]
condition_labels = np.asarray(fit["condition_labels_str"]).astype(str)

print("K:", fit["K"])
print("N participant entries:", len(results_subj))
display(pd.Series(condition_labels).value_counts().reindex(CONDITIONS).to_frame("n"))

example = results_subj[0]
print("sXC shape:", np.asarray(example["sXC"]).shape)
print("S shape:", np.asarray(example["S"]).shape)


## 2. Load HTFA node geometry

In [ ]:

if "centers" in fit and "widths" in fit:
    centers = np.asarray(fit["centers"], dtype=float)
    widths = np.asarray(fit["widths"], dtype=float).ravel()
    print("Using centers/widths saved in fit.")
else:
    if not POSTERIOR_MAT.exists():
        raise FileNotFoundError(f"Missing posterior MAT: {POSTERIOR_MAT}")
    posterior = loadmat(POSTERIOR_MAT)
    centers = np.asarray(posterior["posterior"]["centers"][0][0][0][0][0], dtype=float)
    widths = np.asarray(list(posterior["posterior"]["widths"][0][0][0][0][0][:, 0].T), dtype=float).ravel()
    print("Loaded centers/widths from posterior MAT.")

print("centers:", centers.shape)
print("widths:", widths.shape)


## 3. Map HTFA nodes to Yeo-7 networks

In [ ]:

lookup_table = {
    "Vis": "Visual",
    "SomMot": "Somatomotor",
    "DorsAttn": "Dorsal attention",
    "SalVentAttn": "Ventral attention",
    "Limbic": "Limbic",
    "Cont": "Frontoparietal",
    "Default": "Default mode",
}

NETWORK_ORDER = [
    "Visual", "Somatomotor", "Dorsal attention", "Ventral attention",
    "Limbic", "Frontoparietal", "Default mode",
]

NETWORK_COLORS = {
    "Visual": "#D7DF23",
    "Somatomotor": "#39B54A",
    "Dorsal attention": "#00A79D",
    "Ventral attention": "#27AAE1",
    "Limbic": "#1C75BC",
    "Frontoparietal": "#92278F",
    "Default mode": "#EE2A7B",
}
network_codes = {name: i + 1 for i, name in enumerate(NETWORK_ORDER)}


def nii2cmu(nifti_file, mask_file=None):
    def fullfact(dims):
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        aftervals = np.asmatrix(fullfact(dims[1:]))
        inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
        row = 0
        for i in range(aftervals.shape[0]):
            inds[row:(row + len(vals)), 0] = vals
            inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
            row += len(vals)
        return inds

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        img = nib.load(nifti_file)
        mask = NiftiMasker(mask_strategy="background")
        mask.fit(nifti_file if mask_file is None else mask_file)

    S = img.get_sform()
    Y = np.float32(mask.transform(nifti_file)).copy()
    vmask = np.nonzero(np.array(np.reshape(mask.mask_img_.dataobj, (1, np.prod(mask.mask_img_.shape)), order="C")))[1]
    vox_coords = fullfact(img.shape[0:3])[vmask, ::-1] - 1
    R = np.array(np.dot(vox_coords, S[0:3, 0:3])) + S[:3, 3]
    return {"Y": Y, "R": R}


def rbf(R, center, width):
    return np.exp(-np.sum((R - center) ** 2, axis=1) / width)


def node_labels(centers, widths, networks_cmu):
    labels = []
    for c, w in zip(centers, widths):
        rr = rbf(networks_cmu["R"], c, w)
        label_weights = [
            np.sum(rr[networks_cmu["Y"].ravel() == i])
            for i in range(1, len(NETWORK_ORDER) + 1)
        ]
        labels.append(int(np.argmax(label_weights) + 1))
    return pd.DataFrame({
        "code": labels,
        "Network": [NETWORK_ORDER[i - 1] for i in labels],
    })


if not NILEARN_AVAILABLE:
    raise ImportError("This figure requires nibabel/nilearn to reproduce the HTFA-to-Yeo mapping.")
if not SCHAEFER_TXT.exists():
    raise FileNotFoundError(SCHAEFER_TXT)
if not SCHAEFER_NII.exists():
    raise FileNotFoundError(SCHAEFER_NII)

key = pd.read_csv(
    SCHAEFER_TXT,
    sep="\t",
    header=None,
    names=["id", "name", "x", "y", "z", "t"],
).drop("t", axis=1)
key["network"] = key["name"].apply(lambda x: x.split("_")[2])
key["network"] = key["network"].apply(lambda x: lookup_table[x])
key["code"] = key["network"].apply(lambda x: network_codes[x])
key.set_index("id", inplace=True)
key.loc[0, "code"] = 0

networks_cmu = nii2cmu(str(SCHAEFER_NII))
networks_cmu["Y"] = np.atleast_2d(
    np.array([key.loc[i, "code"] for i in networks_cmu["Y"]]).astype(float)
)
node_codes = node_labels(centers, widths, networks_cmu)

print("Yeo node counts:")
display(node_codes["Network"].value_counts().reindex(NETWORK_ORDER).to_frame("n_nodes"))



## 4. Compute temporal profiles and all-node network composition

For each archetype, the temporal motif is the mean `sXC[:, k]` across participant entries. The spatial map is the mean `S[k, :]` across participant entries. Network composition is computed from the absolute spatial coefficient mass across **all 700 nodes**:

\[
\mathrm{mass}_{k,n}=\frac{\sum_{v\in n}|\overline{S}_{k,v}|}{\sum_v |\overline{S}_{k,v}|}.
\]

The network composition therefore does **not** depend on the display threshold.


In [ ]:

network_rows = []
spatial_maps = {}
temporal_summary = {}

for k in SELECTED_ARCHETYPES:
    tc = np.stack([to_float_array(sub["sXC"])[:, k] for sub in results_subj], axis=0)
    mean_tc = tc.mean(axis=0)
    sem_tc = tc.std(axis=0, ddof=1) / np.sqrt(tc.shape[0])
    temporal_summary[k] = {"mean": mean_tc, "sem": sem_tc}

    coeffs = np.stack([to_float_array(sub["S"])[k, :] for sub in results_subj], axis=0)
    group_map = np.nan_to_num(coeffs.mean(axis=0), nan=0.0, posinf=0.0, neginf=0.0)
    spatial_maps[k] = group_map

    abs_map = np.abs(group_map)
    total = abs_map.sum()
    for network in NETWORK_ORDER:
        mask = node_codes["Network"].to_numpy() == network
        mass = abs_map[mask].sum() / total if total > 0 else np.nan
        network_rows.append({
            "archetype": k,
            "network": network,
            "mass_fraction": mass,
            "n_nodes": int(mask.sum()),
        })

network_mass_df = pd.DataFrame(network_rows)
display(network_mass_df[network_mass_df["archetype"] == SELECTED_ARCHETYPES[0]])


## 5. Cross-fitted top-5 selection frequency

In [ ]:

def parse_selected_archetypes(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return [int(v) for v in x]
    return [int(v) for v in pyast.literal_eval(str(x))]


def selection_frequency_from_runs(run_csv, K, top_m):
    runs = pd.read_csv(run_csv)
    sub = runs[
        (runs["K"].astype(int) == int(K))
        & (runs["top_m"].astype(int) == int(top_m))
        & (runs["condition"].astype(str).isin(CONDITIONS))
    ].copy()

    if len(sub) == 0:
        raise ValueError(f"No K={K} top-{top_m} rows found in {run_csv}")

    rows = []
    for condition in CONDITIONS:
        cur = sub[sub["condition"].astype(str) == condition]
        n_crossfits = len(cur)
        counts = {k: 0 for k in range(K)}
        for selected in cur["selected_archetypes"]:
            for k in parse_selected_archetypes(selected):
                counts[int(k)] += 1
        for k in range(K):
            rows.append({
                "condition": condition,
                "archetype": k,
                "selected_count": counts[k],
                "n_crossfits": n_crossfits,
                "selection_frequency": counts[k] / n_crossfits if n_crossfits > 0 else np.nan,
            })
    return pd.DataFrame(rows)


if HELDOUT_RUN_CSV.exists():
    selection_freq_df = selection_frequency_from_runs(HELDOUT_RUN_CSV, K=K, top_m=TOP_M)
    print("Selection frequency derived from:", HELDOUT_RUN_CSV)
elif HELDOUT_STABILITY_CSV.exists():
    stab = pd.read_csv(HELDOUT_STABILITY_CSV)
    selection_freq_df = stab[(stab["K"].astype(int) == K) & (stab["top_m"].astype(int) == TOP_M)].copy()
    print("Selection frequency loaded from:", HELDOUT_STABILITY_CSV)
else:
    raise FileNotFoundError("Could not find held-out run or stability CSV.")

selected_stability = selection_freq_df[
    selection_freq_df["archetype"].astype(int).isin(SELECTED_ARCHETYPES)
].copy()

pivot_stability = (
    selected_stability
    .pivot_table(index="archetype", columns="condition", values="selection_frequency", aggfunc="mean")
    .reindex(SELECTED_ARCHETYPES)
)
pivot_stability["shared_intact_word"] = pivot_stability[["intact", "word"]].min(axis=1)
display(pivot_stability)


## 6. Build the paper-ready Figure 5

### Upper block: one row per archetype
Each selected archetype gets:
- a **temporal motif** panel (mean ± SEM)
- a **large glass-brain spatial deployment** panel

### Bottom summary block
Instead of repeating small per-row bar charts, the figure now uses two shared summary panels:

1. **Yeo-7 composition across archetypes**  
   A stacked horizontal bar chart, one row per archetype, using the **all-node absolute spatial coefficient mass**.

2. **Cross-fitted top-5 selection stability**  
   A compact heatmap with rows = archetypes and columns = intact / word / rest.

This layout keeps the spatial maps readable while making cross-archetype comparisons easier.


In [ ]:
from PIL import Image, ImageChops
import io


def scale_node_sizes(
    values,
    size_min=18,
    size_max=105,
):
    values = np.asarray(values, dtype=float)

    if len(values) == 0:
        return np.array([])

    vmin = values.min()
    vmax = values.max()

    if np.isclose(vmax, vmin):
        return np.full(
            len(values),
            (size_min + size_max) / 2,
        )

    scaled = (values - vmin) / (vmax - vmin)

    return size_min + scaled * (size_max - size_min)


def render_glass_brain_view(
    coords,
    networks,
    sizes,
    view,
    cmap_lookup=None,
):
    """
    Render one glass-brain view, crop away white margins,
    and return it as an image array.
    """

    if cmap_lookup is None:
        cmap_lookup = NETWORK_COLORS

    tmp_fig = plt.figure(
        figsize=(4.0, 3.4)
    )

    display = niplot.plot_glass_brain(
        None,
        display_mode=view,
        figure=tmp_fig,
        black_bg=False,
        colorbar=False,
        plot_abs=False,
        annotate=True,
    )

    for network in NETWORK_ORDER:

        mask = (
            networks == network
        )

        if np.any(mask):

            display.add_markers(
                marker_coords=coords[mask],
                marker_color=cmap_lookup[network],
                marker_size=sizes[mask].tolist(),
            )

    buf = io.BytesIO()

    tmp_fig.savefig(
        buf,
        format="png",
        dpi=220,
        facecolor="white",
        bbox_inches="tight",
        pad_inches=0.02,
    )

    plt.close(tmp_fig)

    buf.seek(0)

    img = Image.open(buf).convert("RGB")

    # --------------------------------------------------------
    # Remove remaining white border
    # --------------------------------------------------------
    white = Image.new(
        "RGB",
        img.size,
        "white",
    )

    diff = ImageChops.difference(
        img,
        white,
    ).convert("L")

    diff = diff.point(
        lambda p: 255 if p > 8 else 0
    )

    bbox = diff.getbbox()

    if bbox is not None:
        img = img.crop(bbox)

    arr = np.asarray(img)

    buf.close()

    return arr


# ============================================================
# SUMMARY TABLES
# ============================================================

composition_table = (
    network_mass_df
    .pivot_table(
        index="archetype",
        columns="network",
        values="mass_fraction",
        aggfunc="first",
    )
    .reindex(
        index=SELECTED_ARCHETYPES,
        columns=NETWORK_ORDER,
    )
)

stability_heatmap = (
    selected_stability
    .pivot_table(
        index="archetype",
        columns="condition",
        values="selection_frequency",
        aggfunc="mean",
    )
    .reindex(
        index=SELECTED_ARCHETYPES,
        columns=CONDITIONS,
    )
)


# ============================================================
# MAIN FIGURE
# ============================================================

fig = plt.figure(
    figsize=(17, 22)
)

outer = fig.add_gridspec(
    2,
    1,
    height_ratios=[6.4, 1.8],
    hspace=0.22,
)


# ============================================================
# TOP: ARCHETYPE ROWS
# temporal | left brain | axial brain | right brain
# ============================================================

top = outer[0].subgridspec(
    len(SELECTED_ARCHETYPES),
    4,
    width_ratios=[
        1.55,
        1.15,
        0.90,
        1.15,
    ],
    wspace=0.06,
    hspace=0.34,
)


for row_i, k in enumerate(
    SELECTED_ARCHETYPES
):

    # ========================================================
    # TEMPORAL MOTIF
    # ========================================================

    ax_t = fig.add_subplot(
        top[row_i, 0]
    )

    mean_tc = temporal_summary[k]["mean"]
    sem_tc = temporal_summary[k]["sem"]

    t = np.arange(
        len(mean_tc)
    )

    ax_t.plot(
        t,
        mean_tc,
        linewidth=1.8,
    )

    ax_t.fill_between(
        t,
        mean_tc - sem_tc,
        mean_tc + sem_tc,
        alpha=0.20,
    )

    ax_t.axhline(
        0,
        linewidth=0.7,
        linestyle=":",
    )

    ax_t.set_xlim(
        0,
        len(mean_tc) - 1,
    )

    ax_t.set_ylabel(
        f"A{k}",
        rotation=0,
        labelpad=28,
        fontsize=14,
        fontweight="bold",
        va="center",
    )

    ax_t.tick_params(
        axis="both",
        labelsize=12,
    )

    if row_i == (
        len(SELECTED_ARCHETYPES) - 1
    ):

        ax_t.set_xlabel(
            "Timepoint",
            fontsize=14,
        )

    else:

        ax_t.set_xticklabels([])

    if row_i == 0:

        ax_t.set_title(
            "Temporal motif",
            fontweight="bold",
            fontsize=15,
            pad=10,
        )


    # ========================================================
    # PREPARE SPATIAL MAP
    # ========================================================

    group_map = spatial_maps[k]

    display_max = np.max(
        group_map
    )

    keep = (
        group_map
        >= DISPLAY_THR_FRAC
        * display_max
    )

    coords = centers[
        keep
    ]

    vals = group_map[
        keep
    ]

    nets = (
        node_codes
        .loc[
            keep,
            "Network",
        ]
        .to_numpy()
    )

    sizes = scale_node_sizes(
        vals,
        size_min=18,
        size_max=105,
    )


    # ========================================================
    # THREE BRAIN VIEWS
    # ========================================================

    views = [
        "l",
        "z",
        "r",
    ]

    brain_axes = []

    for col_i, view in enumerate(
        views,
        start=1,
    ):

        ax_b = fig.add_subplot(
            top[
                row_i,
                col_i,
            ]
        )

        brain_axes.append(
            ax_b
        )

        brain_img = (
            render_glass_brain_view(
                coords=coords,
                networks=nets,
                sizes=sizes,
                view=view,
                cmap_lookup=NETWORK_COLORS,
            )
        )

        ax_b.imshow(
            brain_img,
            interpolation="antialiased",
        )

        ax_b.set_axis_off()

        ax_b.set_anchor(
            "C"
        )

    if row_i == 0:

        brain_axes[1].set_title(
            "Spatial deployment",
            fontweight="bold",
            fontsize=15,
            pad=10,
        )


# ============================================================
# BOTTOM SUMMARY PANELS
# ============================================================

bottom = outer[1].subgridspec(
    1,
    2,
    width_ratios=[
        1.70,
        1.00,
    ],
    wspace=0.18,   # pulls right panel left
)


# ============================================================
# YEO-7 COMPOSITION
# ============================================================

ax_comp = fig.add_subplot(
    bottom[0, 0]
)

ypos = np.arange(
    len(
        SELECTED_ARCHETYPES
    )
)

left = np.zeros(
    len(
        SELECTED_ARCHETYPES
    )
)

for network in NETWORK_ORDER:

    vals = (
        composition_table[
            network
        ]
        .to_numpy(
            dtype=float
        )
    )

    ax_comp.barh(
        ypos,
        vals,
        left=left,
        color=NETWORK_COLORS[
            network
        ],
        edgecolor="none",
        height=0.72,
        label=network,
    )

    left = left + vals


ax_comp.set_yticks(
    ypos
)

ax_comp.set_yticklabels(
    [
        f"A{k}"
        for k
        in SELECTED_ARCHETYPES
    ],
    fontsize=15,
)

ax_comp.invert_yaxis()

ax_comp.set_xlim(
    0,
    1,
)

ax_comp.set_xlabel(
    "Fraction of |spatial weight|",
    fontsize=15,
    labelpad=8,
)

ax_comp.tick_params(
    axis="x",
    labelsize=15,
)

ax_comp.set_title(
    "Yeo-7 composition across selected archetypes",
    fontweight="bold",
    fontsize=17,
)


# ============================================================
# SELECTION STABILITY
# ============================================================

ax_heat = fig.add_subplot(
    bottom[0, 1]
)

heat = (
    stability_heatmap
    .to_numpy(
        dtype=float
    )
)

im = ax_heat.imshow(
    heat,
    aspect="auto",
    vmin=0,
    vmax=1,
)


ax_heat.set_xticks(
    np.arange(
        len(CONDITIONS)
    )
)

ax_heat.set_xticklabels(
    [
        "Intact",
        "Word",
        "Rest",
    ],
    fontsize=15,
)

ax_heat.set_yticks(
    np.arange(
        len(
            SELECTED_ARCHETYPES
        )
    )
)

ax_heat.set_yticklabels(
    [
        f"A{k}"
        for k
        in SELECTED_ARCHETYPES
    ],
    fontsize=15,
)

ax_heat.set_title(
    "Cross-fitted top-5 selection stability",
    fontweight="bold",
    fontsize=17,
)


# ------------------------------------------------------------
# Heatmap cell values
# ------------------------------------------------------------

for r in range(
    heat.shape[0]
):

    for c in range(
        heat.shape[1]
    ):

        val = heat[
            r,
            c,
        ]

        txt_color = (
            "white"
            if val >= 0.55
            else "black"
        )

        ax_heat.text(
            c,
            r,
            f"{val:.2f}",
            ha="center",
            va="center",
            fontsize=13,
            color=txt_color,
        )


# ------------------------------------------------------------
# Colorbar
# ------------------------------------------------------------

cbar = fig.colorbar(
    im,
    ax=ax_heat,
    fraction=0.046,
    pad=0.04,
)

cbar.set_label(
    "Selection frequency",
    fontsize=15,
)

cbar.ax.tick_params(
    labelsize=14,
)


# ============================================================
# SHARED YEO LEGEND
# ============================================================

legend_handles = [

    plt.Rectangle(
        (0, 0),
        1,
        1,
        color=NETWORK_COLORS[
            network
        ],
    )

    for network
    in NETWORK_ORDER
]


fig.legend(
    legend_handles,
    NETWORK_ORDER,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=14,
    bbox_to_anchor=(
        0.5,
        0.018,   # moved lower, away from x-axis label
    ),
)


# ============================================================
# TITLES / SAVE
# ============================================================

fig.suptitle(
    (
        "Held-out-selected spatial "
        f"archetypes at K={K}"
    ),
    fontsize=19,
    fontweight="bold",
    y=0.995,
)


fig.subplots_adjust(
    bottom=0.115,   # extra room for legend
    top=0.965,
)


fig.savefig(
    FIGURE_PNG,
    dpi=DPI,
    bbox_inches="tight",
)


fig.savefig(
    FIGURE_PDF,
    bbox_inches="tight",
)


plt.show()


print(
    "Saved:",
    FIGURE_PNG,
)

print(
    "Saved:",
    FIGURE_PDF,
)

## 7. Export the underlying Figure 5 data

In [ ]:

network_mass_df.to_csv(OUTPUT_DIR / "Figure5_Yeo7_allnode_mass.csv", index=False)
selected_stability.to_csv(OUTPUT_DIR / "Figure5_crossfit_selection_frequency.csv", index=False)

spatial_rows = []
for k in SELECTED_ARCHETYPES:
    group_map = spatial_maps[k]
    for node_i, weight in enumerate(group_map):
        spatial_rows.append({
            "archetype": k,
            "node": node_i,
            "x": centers[node_i, 0],
            "y": centers[node_i, 1],
            "z": centers[node_i, 2],
            "network": node_codes.loc[node_i, "Network"],
            "mean_spatial_weight": weight,
        })
pd.DataFrame(spatial_rows).to_csv(OUTPUT_DIR / "Figure5_spatial_maps_allnodes.csv", index=False)

temporal_rows = []
for k in SELECTED_ARCHETYPES:
    mean_tc = temporal_summary[k]["mean"]
    sem_tc = temporal_summary[k]["sem"]
    for t, (mean_v, sem_v) in enumerate(zip(mean_tc, sem_tc)):
        temporal_rows.append({
            "archetype": k,
            "timepoint": t,
            "mean_temporal_profile": mean_v,
            "sem_temporal_profile": sem_v,
        })
pd.DataFrame(temporal_rows).to_csv(OUTPUT_DIR / "Figure5_temporal_profiles.csv", index=False)

print("Exported all Figure 5 source tables to:", OUTPUT_DIR.resolve())


## 8. Draft figure caption

**Figure 5. Held-out-selected spatial archetypes reveal interpretable distributed motifs.**
Five spatial archetypes were selected from the representative \(K=46\) across-condition solution based on their cross-fitted top-5 selection stability in the stimulus conditions. **Top:** each row shows one selected archetype (A4, A5, A6, A12, and A8), including its cross-participant mean temporal profile with SEM and its participant-averaged spatial coefficient map rendered as a glass-brain view. Node color indicates Yeo-7 network membership and node size reflects coefficient magnitude among nodes retained for visualization. The display threshold is used **for visualization only**. **Bottom left:** stacked bars summarize each archetype's Yeo-7 composition, quantified as the fraction of absolute spatial coefficient mass assigned to each network across **all** HTFA nodes. **Bottom right:** heatmap showing cross-fitted top-5 selection frequency across intact, word-scrambled, and rest conditions.

**Important:** the Yeo composition shown here is descriptive. Statistical claims of network enrichment should be based on the separate network-size-preserving null analysis.

### Results-language


> At the representative intermediate-resolution solution (\(K=46\)), held-out sparse selection identified five frequently selected spatial archetypes with distinct temporal profiles and distributed spatial deployment across canonical functional systems.

Do not describe any network as *enriched* from this figure alone.
